# Lag-Llama Research Experiment: Zero-Shot Forecasting

This notebook evaluates the **Lag-Llama** foundation model on the ETT (ETTh1, ETTh2, ETTm1, ETTm2), Weather, and Electricity datasets.

### Metrics Evaluated:
1. **MSE** (Mean Squared Error)
2. **MAE** (Mean Absolute Error)
3. **Hallucination Rate**: Defined as the percentage of predicted values that deviate by more than 3 standard deviations from the context distribution.
4. **Perplexity**: Calculated as the exponent of the Negative Log-Likelihood (NLL) of the predicted distribution.

### Datasets:
Please upload the following CSV files to the `/content/` directory in Colab:
- `ETTh1.csv`, `ETTh2.csv`, `ETTm1.csv`, `ETTm2.csv` (Hourly and 15-min ETT datasets)
- `weather.csv` (Weather dataset)
- `electricity.csv` (Electricity dataset)

---

## 1. Setup and Installation

In [ ]:
!git clone https://github.com/time-series-foundation-models/lag-llama.git
%cd lag-llama
!pip install -r requirements.txt
!pip install gluonts==0.16.0
!huggingface-cli download time-series-foundation-models/Lag-Llama lag-llama.ckpt --local-dir /content/lag-llama

Cloning into 'lag-llama'...
remote: Enumerating objects: 508, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 508 (delta 155), reused 114 (delta 114), pack-reused 325 (from 3)
Receiving objects: 100% (508/508), 286.88 KiB | 11.03 MiB/s, done.
Resolving deltas: 100% (253/253), done.
/content/lag-llama
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.7 MB/s eta 0:00:0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 41.3 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
^C
/bin/bash: line 1: huggingface-cli: command not found


In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from gluonts.dataset.pandas import PandasDataset
from gluonts.evaluation import make_evaluation_predictions, Evaluator
from lag_llama.gluon.estimator import LagLlamaEstimator
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'lag_llama'

## 2. Define Evaluation Metrics

In [ ]:
def calculate_hallucination_rate(forecasts, tss, threshold_sigma=3):
    """
    Calculate hallucination rate: % of predicted values that are 'unrealistic'
    relative to the context distribution.
    """
    hallucinations = 0
    total_points = 0

    for forecast, ts in zip(forecasts, tss):
        # Context is everything before the forecast start
        context = ts[:-len(forecast.samples[0])].values.flatten()
        mean_ctx = np.mean(context)
        std_ctx = np.std(context)

        # Predicted values (using median)
        pred = np.median(forecast.samples, axis=0)

        # Check for values outside threshold
        outliers = np.sum((pred > mean_ctx + threshold_sigma * std_ctx) |
                          (pred < mean_ctx - threshold_sigma * std_ctx))

        hallucinations += outliers
        total_points += len(pred)

    return hallucinations / total_points if total_points > 0 else 0

def calculate_perplexity(agg_metrics):
    """
    Calculate perplexity as exp(NLL).
    GluonTS provides NegativeLogLikelihood in its evaluation metrics.
    """
    nll = agg_metrics.get("NegativeLogLikelihood", np.nan)
    return np.exp(nll) if not np.isnan(nll) else np.nan

## 3. Experiment Execution Function

In [ ]:
def run_lag_llama_experiment(dataset_path, prediction_length=24, context_length=32):
    dataset_name = os.path.basename(dataset_path).split('.')[0]
    print(f"\n--- Evaluating {dataset_name} ---")

    # Load data
    df = pd.read_csv(dataset_path)
    # Standard ETT/Weather/Electricity format: 'date' and multiple columns
    # We use the last column as the target for univariate forecasting
    target_col = df.columns[-1]
    df['date'] = pd.to_datetime(df['date'])

    # Prepare GluonTS dataset
    dataset = PandasDataset(df, target=target_col, timestamp="date", freq=pd.infer_freq(df['date']))

    ckpt_path = "lag-llama.ckpt"
    ckpt = torch.load(ckpt_path, map_location=device)
    estimator_args = ckpt["hyper_parameters"]["model_kwargs"]

    estimator = LagLlamaEstimator(
        ckpt_path=ckpt_path,
        prediction_length=prediction_length,
        context_length=context_length,
        input_size=estimator_args["input_size"],
        n_layer=estimator_args["n_layer"],
        n_embd_per_head=estimator_args["n_embd_per_head"],
        n_head=estimator_args["n_head"],
        scaling=estimator_args["scaling"],
        time_feat=estimator_args["time_feat"],
        batch_size=64,
        num_parallel_samples=100,
        device=device,
    )

    lightning_module = estimator.create_lightning_module()
    transformation = estimator.create_transformation()
    predictor = estimator.create_predictor(transformation, lightning_module)

    forecast_it, ts_it = make_evaluation_predictions(
        dataset=dataset,
        predictor=predictor,
        num_samples=100
    )

    forecasts = list(forecast_it)
    tss = list(ts_it)

    evaluator = Evaluator(quantiles=[0.1, 0.5, 0.9])
    agg_metrics, item_metrics = evaluator(tss, forecasts)

    results = {
        "Dataset": dataset_name,
        "MSE": agg_metrics["MSE"],
        "MAE": agg_metrics["MAE"],
        "Hallucination Rate": calculate_hallucination_rate(forecasts, tss),
        "Perplexity": calculate_perplexity(agg_metrics)
    }

    for k, v in results.items():
        if isinstance(v, float): print(f"{k}: {v:.4f}")
        else: print(f"{k}: {v}")

    return results

## 4. Run Experiments on All Datasets

Ensure you have uploaded the files to `/content/`.

In [ ]:
datasets = [
    "/content/ETTh1.csv", "/content/ETTh2.csv",
    "/content/ETTm1.csv", "/content/ETTm2.csv",
    "/content/weather.csv", "/content/electricity.csv"
]

all_results = []
for ds_path in datasets:
    if os.path.exists(ds_path):
        res = run_lag_llama_experiment(ds_path)
        all_results.append(res)
    else:
        print(f"Skipping {ds_path}: File not found.")

if all_results:
    results_df = pd.DataFrame(all_results)
    display(results_df)
    results_df.to_csv("/content/experiment_results.csv", index=False)